# Review the Clusters and Freeze the Codebook

| Item | Description |
|---|---|
| Goal | Review the 150 drafted clusters of T2.1.3 and freeze the codebook: keep, merge, or drop each cluster for a stated reason, relabel eight, and record the result as codebook v1.0 with 111 topics. Then assign every Track 1 phrase to its topic by cluster membership, give each hearing its primary topic, and save the topic centroids that T2.2.6 places the stance records against. |
| Input | `codebook_draft.parquet`, `phrases.parquet`, `cut_labels.parquet`, and `unit_embeddings.npz` in `taxonomy/` of the T2.1 Interim folder; `track1/topics.parquet` from T2.1.1. |
| Output | In the T2.1 Processed folder: `codebook_v1.parquet` (frozen, SHA-256 recorded), `track1_assigned.parquet` (one row per topic phrase per hearing with its topic), `hearing_topics_v1.parquet` (one row per hearing with its primary topic), and `topic_centroids.npz` (one unit vector per topic). Under `R - Results/tables/...`: `table_codebook_v1.csv` and `.tex`, and `table_codebook_v1.md`, the review record with every decision and its reason. |
| Variables | • `topic_id`: the cluster number of a kept topic; the numbers are not consecutive, because dropped and merged clusters keep their numbers out of the codebook<br>• `theme_id`, `theme_label`: the 15-cluster level, kept in the frozen file for the record; the analysis uses the hand-made groups of T2.1.5 instead<br>• `primary_label`: the topic with the most phrases in the hearing; a tie goes to the lower topic number |
| Run | First run 2026-08-22, which froze the codebook (SHA-256 `fd81e781…`). Rebuilt into a scratch copy on 2026-10-31: the three tables came out identical and the rebuilt codebook file reproduces the frozen SHA-256 byte for byte; the outputs shown are from that rebuild. The stored August files stay the inputs of the later steps; the centroids and the tables are new outputs written on 2026-10-31. |


## Step 1. Import packages


In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T2 - Topic Extraction and Stance Analysis/T2.1 - Track 1 - Topic Extraction based on Hearing Summaries")
NOTEBOOK_NAME = "T2.1.4 - Freeze - Review the Clusters and Freeze the Codebook"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
TRACK1_FOLDER = INTERIM_FOLDER / "track1"
TAXONOMY_FOLDER = INTERIM_FOLDER / "taxonomy"
K_TOPIC, K_THEME = 150, 15
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Record the review decisions

Every decision carries its reason. Drops are families of a shared word rather than a shared subject, such as "program" or "initiative". Mixtures of unrelated subjects and fragments of fewer than four phrases are dropped too. Merges join a cluster to the topic it belongs to. Relabels replace a drafted label that misnamed its cluster.


In [4]:
draft = pd.read_parquet(TAXONOMY_FOLDER / "codebook_draft.parquet")
topics_d = draft[draft["kind"] == "topic"].set_index("cluster")
themes_d = draft[draft["kind"] == "theme"].set_index("cluster")

DROP = {
 104: "surface family of named Acts across domains (Jones/GLBA/FAST/FIRST)",
 82: "legal-clause family (First Amendment/grandfather/Section 13(b))",
 88: "grab-bag: auctions+MOU+pricing (s0.24, c0.55)",
 96: "'standards' word family across domains (NIST/CAFE/forensic)",
 97: "grab-bag: Yucca+data brokers+Form 477",
 98: "heterogeneous internet terms (interoperability/IoT/ICANN)",
 99: "census + cost-benefit mixture",
 123: "'program' word family (BEAD/I-Corps/NPOESS/TWIC)",
 70: "'partnership' word family (PPP/Bayh-Dole/Screening)",
 72: "'initiative' word family (Lands Legacy/SunShot/BRAIN)",
 124: "'assistance program' mixture (E-Rate/ACP/Visa Waiver/Paycheck)",
 57: "'America act' surface family (Connect/GROW/Brand USA/Jobs Plan)",
 28: "'reduction' word family (IRA/GHG/Deficit Reduction)",
 71: "planning-office fragments (JPDO), incoherent remainder",
 113: "research-program mixture (Low Dose/CRS/Engineering), s0.27",
 51: "mixed telephone/alert terms",
 77: "regulatory-framework grab-bag (codes/Fast Track/NGB/OIRA)",
 85: "tax/carbon mixture (RGGI/social cost/gas tax/HMT)",
 80: "licensing-compliance grab-bag (retransmission/APA/BOD), s0.28",
 81: "incoherent mixture (cross-ownership/quarantine/ADS-B)",
 106: "incoherent mixture (FISMA/ADS-B/Atlas V), s0.25 c0.54",
 140: "'-Gen' surface family (NextGen aviation + FutureGen coal)",
 75: "'reform' word family (immigration/CONCACAF/CPB/MAP-21)",
 63: "1 phrase", 131: "2 phrases", 66: "2 phrases", 135: "3 junk phrases",
 39: "3 junk phrases (Cuba aid/Katrina)", 101: "5 mixed phrases",
 142: "5 mixed phrases",
}
MERGE = {
 105: (32, "flow valves/blowout preventers = pipeline safety equipment"),
 33: (78, "Clean Air/Water Act phrases belong with EPA regulation"),
 64: (65, "SAFE DATA/WEB acts join the data-protection/spam cluster"),
 8: (6, "ABI is a GOES instrument -> weather-satellite topic"),
 5: (76, "boxing/sports acts join the sports-integrity cluster"),
 31: (44, "Magnuson-Stevens act family joins fisheries management"),
 42: (41, "firefighter grants join fire/emergency response"),
 16: (121, "Noyce teacher scholarships join STEM education"),
 118: (95, "NITRD joins advanced-computing programs"),
}
RELABEL = {
 74: "inspectors general and federal accountability",
 86: "passenger fees and charges",
 95: "advanced computing and ARPA-E programs",
 108: "stem cell research and bioethics",
 116: "science competitiveness legislation",
 133: "surface transportation agencies",
 114: "science policy bodies (OSTP and advisory boards)",
 138: "public alert and risk information systems",
}
KEEP = [c for c in topics_d.index if c not in DROP and c not in MERGE]
assert len(KEEP) + len(DROP) + len(MERGE) == 150
print(f"KEEP {len(KEEP)} / MERGE {len(MERGE)} / DROP {len(DROP)}")


KEEP 111 / MERGE 9 / DROP 30


## Step 4. Build and freeze codebook v1.0

Each kept topic takes its drafted label, definition, exclusion notes, and up to five example phrases topped up from its members. The 15-cluster level is attached by the plurality of phrase weight and kept for the record. The file's SHA-256 is the freeze mark.


In [5]:
phrases = pd.read_parquet(TAXONOMY_FOLDER / "phrases.parquet")
cuts = pd.read_parquet(TAXONOMY_FOLDER / "cut_labels.parquet")
ph = phrases[phrases["family_id"] >= 0].merge(cuts[["family_id", f"K{K_TOPIC}", f"K{K_THEME}"]], on="family_id")
ph = ph.rename(columns={f"K{K_TOPIC}": "topic_c", f"K{K_THEME}": "theme_c"})
remap = {c: c for c in KEEP}
remap.update({c: t for c, (t, _) in MERGE.items()})
ph["topic_id"] = ph["topic_c"].map(lambda c: remap.get(c, -1))

nest = (ph[ph["topic_id"] >= 0].groupby(["topic_id", "theme_c"])["weight"].sum().reset_index()
        .sort_values("weight", ascending=False).drop_duplicates("topic_id").set_index("topic_id")["theme_c"])

rows = []
for cid in KEEP:
    r = topics_d.loc[cid]
    e = json.loads(r["entry"])
    members = json.loads(r["top_members"])
    ex = list(dict.fromkeys(e["exemplars"]))
    for m in members:
        if len(ex) >= 5:
            break
        if m.casefold() not in {x.casefold() for x in ex}:
            ex.append(m)
    th = int(nest.loc[cid])
    te = json.loads(themes_d.loc[th, "entry"]) if themes_d.loc[th, "entry"] else None
    rows.append({
        "topic_id": int(cid), "theme_id": th, "theme_label": te["label"] if te else f"theme {th}",
        "label": RELABEL.get(cid, e["label"]), "definition": e["definition"],
        "exclusion_notes": json.dumps(e["exclusion_notes"]), "exemplars": json.dumps(ex[:5]),
        "stability": float(r["stability"]), "cohesion": float(r["cohesion"]), "weight": float(r["weight"]), "n_phrases": int(r["n_phrases"]),
    })
codebook = pd.DataFrame(rows).sort_values(["theme_id", "topic_id"])
codebook.to_parquet(PROCESSED_FOLDER / "codebook_v1.parquet", index=False)
sha = hashlib.sha256((PROCESSED_FOLDER / "codebook_v1.parquet").read_bytes()).hexdigest()
print(f"codebook v1.0: {len(codebook)} topics / {codebook['theme_id'].nunique()} themes")
print(f"FREEZE sha256: {sha}")


codebook v1.0: 111 topics / 14 themes
FREEZE sha256: fd81e7814be416617cb52ac4bfaee3eec8086192aa51fda171342b9bb1919abc


## Step 5. Assign the Track 1 phrases and name each hearing's primary topic

A phrase belongs to the topic of its cluster. The primary topic of a hearing is the topic with the most phrases among its up to five. A tie goes to the lower topic number. The topic centroids, the weighted mean embedding of each topic's phrases, are saved for T2.2.6.


In [6]:
meta = codebook.set_index("topic_id")
key2topic = ph.set_index("key")["topic_id"]

t1_raw = pd.read_parquet(TRACK1_FOLDER / "topics.parquet")
t1_rows = []
for pkg, year, chamber, js in zip(t1_raw["packageId"], t1_raw["year"], t1_raw["chamber"], t1_raw["topics_json"]):
    for t in json.loads(js):
        t1_rows.append({"packageId": pkg, "year": year, "chamber": chamber, "rank": t["rank"], "phrase": t["topic"], "key": t["topic"].casefold().strip()})
t1 = pd.DataFrame(t1_rows)
t1["topic_id"] = t1["key"].map(key2topic).fillna(-1).astype(int)
t1["topic_label"] = t1["topic_id"].map(meta["label"]).fillna("other")
t1.drop(columns=["key"]).to_parquet(PROCESSED_FOLDER / "track1_assigned.parquet", index=False)
print(f"track1 mentions in codebook topics: {(t1['topic_id'] >= 0).mean():.1%}")

counts = t1[t1["topic_id"] >= 0].groupby(["packageId", "topic_id"]).size().rename("n").reset_index()
modal = counts.sort_values(["packageId", "n"], ascending=[True, False]).drop_duplicates("packageId")
modal["primary_label"] = modal["topic_id"].map(meta["label"])
modal["primary_theme"] = modal["topic_id"].map(meta["theme_label"])
modal.to_parquet(PROCESSED_FOLDER / "hearing_topics_v1.parquet", index=False)
print(f"hearings with a primary topic: {modal['packageId'].nunique():,} / {t1['packageId'].nunique():,}")

dat = np.load(TAXONOMY_FOLDER / "unit_embeddings.npz", allow_pickle=True)
X = dat["X"].astype(np.float32)
nk = {n: i for i, n in enumerate(dat["norm_key"])}
ph["ux"] = ph["norm"].str.casefold().map(nk)
tids = sorted(meta.index)
C = np.zeros((len(tids), X.shape[1]), dtype=np.float32)
for k, tid in enumerate(tids):
    g = ph[ph["topic_id"] == tid]
    v = (X[g["ux"].to_numpy()] * g["weight"].to_numpy()[:, None]).sum(0)
    C[k] = v / np.linalg.norm(v)
np.savez_compressed(PROCESSED_FOLDER / "topic_centroids.npz", C=C, topic_id=np.array(tids))
print(f"topic centroids: {C.shape[0]} topics x {C.shape[1]} dimensions -> topic_centroids.npz")


track1 mentions in codebook topics: 83.2%
hearings with a primary topic: 2,342 / 2,352


topic centroids: 111 topics x 1024 dimensions -> topic_centroids.npz


## Step 6. Write the codebook table and the review record

The table lists the 111 topics with their labels, definitions, and diagnostics. The record lists every drop, merge, and relabel with its reason, then the frozen codebook.


In [7]:
TABLE_FOLDER.mkdir(parents=True, exist_ok=True)
table = codebook[["topic_id", "label", "definition", "theme_id", "theme_label", "stability", "cohesion", "weight", "n_phrases"]].sort_values("topic_id")
table.to_csv(TABLE_FOLDER / "table_codebook_v1.csv", index=False)
(table[["topic_id", "label", "n_phrases", "stability", "cohesion"]].style.hide(axis="index").format({"stability": "{:.2f}", "cohesion": "{:.2f}"})
 .to_latex(TABLE_FOLDER / "table_codebook_v1.tex", hrules=True, column_format="rlrrr", environment="longtable"))

lines = ["# Codebook v1.0, reviewed and frozen (2026-08-22)", "", f"Freeze SHA-256: `{sha}`", "",
         f"KEEP {len(KEEP)} / MERGE {len(MERGE)} / DROP {len(DROP)}; stability and cohesion are reported diagnostics and gate nothing. "
         "The review was made on the draft of T2.1.3 without reference to the CAP codes.", "", "## Drops", ""]
lines += [f"- Topic {c}: {r}" for c, r in sorted(DROP.items())]
lines += ["", "## Merges", ""]
lines += [f"- Topic {c} -> {t}: {r}" for c, (t, r) in sorted(MERGE.items())]
lines += ["", "## Relabels", ""]
lines += [f"- Topic {c}: '{l}'" for c, l in sorted(RELABEL.items())]
lines += ["", "## Frozen codebook", ""]
for th, g in codebook.groupby("theme_id"):
    lines += [f"### Broad cluster {th}: {g['theme_label'].iloc[0]}", ""]
    for r in g.itertuples():
        lines += [f"- **{r.topic_id} {r.label}** (stability {r.stability:.2f}, cohesion {r.cohesion:.2f}, weight {r.weight:.0f}): {r.definition}"]
    lines += [""]
(TABLE_FOLDER / "table_codebook_v1.md").write_text("\n".join(lines), encoding="utf-8")
print(f"wrote table_codebook_v1.csv/.tex ({len(table)} rows) and table_codebook_v1.md ({len(lines):,} lines)")


wrote table_codebook_v1.csv/.tex (111 rows) and table_codebook_v1.md (217 lines)


## Step 7. Check the saved files

- 111 unique topics in the codebook, as many as kept clusters.
- Every Track 1 phrase has a topic or the other pool; every hearing with an assigned phrase has a primary topic.
- One unit-length centroid per topic.
- The content hash is the SHA-256 of the sorted table written as CSV. It identifies the codebook's content; the file hash depends also on the parquet writer version.


In [8]:
back = pd.read_parquet(PROCESSED_FOLDER / "codebook_v1.parquet")
assert back["topic_id"].is_unique and len(back) == len(KEEP)
t1b = pd.read_parquet(PROCESSED_FOLDER / "track1_assigned.parquet")
assert set(t1b["topic_id"]) <= set(back["topic_id"]) | {-1}
cent = np.load(PROCESSED_FOLDER / "topic_centroids.npz")
assert list(cent["topic_id"]) == sorted(back["topic_id"]) and np.allclose(np.linalg.norm(cent["C"], axis=1), 1, atol=1e-4)
content_sha = hashlib.sha256(back.sort_values("topic_id").to_csv(index=False).encode("utf-8")).hexdigest()
print(f"1) codebook : {len(back)} topics / {back['theme_id'].nunique()} broad clusters; file SHA-256 {sha[:16]}..., content SHA-256 {content_sha[:16]}...")
print(f"2) coverage : {(t1b['topic_id'] >= 0).mean():.1%} of Track 1 phrases in a codebook topic; {modal['packageId'].nunique():,} hearings with a primary topic")
print(f"3) centroids: {cent['C'].shape[0]} unit vectors of {cent['C'].shape[1]} dimensions")


1) codebook : 111 topics / 14 broad clusters; file SHA-256 fd81e7814be41661..., content SHA-256 a57b3667fcc15520...
2) coverage : 83.2% of Track 1 phrases in a codebook topic; 2,342 hearings with a primary topic
3) centroids: 111 unit vectors of 1024 dimensions
